# Tutorial 28: Decision Models

LangChain 1.5 adds decision models for questions that have a typed answer rather than a paragraph. A decision model looks at some state and returns:

| Question | Answer | Use |
|---|---|---|
| `Noul` | Probability from 0 to 1 | Yes or no, such as "is this tool call risky?" |
| `Choice` | One label, plus a probability for every label | Routing |
| `Score` | A position on an ordered rubric | Severity |
| `Box` | A region of an image | Not used in this notebook |

`ModelRouterMiddleware` and `AutoModeMiddleware` are beta middleware that call a decision model for you. Both require `classifier=`. The hosted classifier is `TypeSafeClassifier` from `langchain-typesafe`, which needs `TYPESAFE_API_KEY`. This notebook implements a local decision model so the routing and the block are deterministic and run with no key.


## 1. A local decision model


In [1]:
import warnings
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import (
    AutoModeMiddleware,
    ModelChoice,
    ModelRouterMiddleware,
)
from langchain_core.decisions import (
    BaseDecisionModel,
    Choice,
    ChoiceAnswer,
    ClassifierResponse,
    Noul,
    NoulAnswer,
    Score,
    ScoreAnswer,
)
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool

warnings.filterwarnings("ignore", message=".*in beta.*")


class ScriptedChat(GenericFakeChatModel):
    """Returns canned replies, including tool calls, and never calls an API."""

    def bind_tools(self, tools, **kwargs):
        return self


class KeywordDecisionModel(BaseDecisionModel):
    """Stand-in for TypeSafeClassifier.

    A hosted decision model reads the question instructions. This one matches
    keywords so each cell has an obvious result.
    """

    model: str = "keyword"

    @property
    def _provider(self) -> str:
        return "local"

    def _classify(self, state: Any, questions: dict):
        text = state if isinstance(state, str) else str(state).lower()
        answers = {}
        for key, question in questions.items():
            if isinstance(question, Choice):
                labels = list(question.criteria)
                picked = labels[0]
                if "powerful" in labels and "architecture" in text:
                    picked = "powerful"
                elif "infra" in labels and any(word in text for word in ("500", "deploy", "outage")):
                    picked = "infra"
                else:
                    for label in labels:
                        if label.lower() in text:
                            picked = label
                            break
                answers[key] = ChoiceAnswer(
                    type="choice",
                    choice=picked,
                    probabilities={label: (0.9 if label == picked else 0.1) for label in labels},
                    confidence=0.9,
                )
            elif isinstance(question, Noul):
                risky = any(word in text for word in ("delete", "outage", "500"))
                answers[key] = NoulAnswer(type="noul", noul=0.95 if risky else 0.05)
            elif isinstance(question, Score):
                level = 2 if any(word in text for word in ("outage", "500")) else 0
                answers[key] = ScoreAnswer(
                    type="score",
                    score=float(level),
                    legend={index: criterion for index, criterion in enumerate(question.criteria)},
                    probabilities={index: (0.8 if index == level else 0.1) for index in range(len(question.criteria))},
                    confidence=0.8,
                )
            else:
                raise TypeError(f"This tutorial does not answer {type(question).__name__}")
        return ClassifierResponse(model=self.model, answers=answers)


classifier = KeywordDecisionModel()
print("Local decision model ready.")


Local decision model ready.


## 2. Ask directly

`invoke` takes `state` and `questions`. The same state is scored against every question. Read `response.nouls`, `response.choices`, and `response.scores`.


In [2]:
incident = "The deploy failed twice and customers are seeing 500s."
calm = "Where is the pricing page?"

for label, text in (("incident", incident), ("calm", calm)):
    response = classifier.invoke({
        "state": text,
        "questions": {
            "urgent": Noul(instructions="Does this need attention right now?"),
            "team": Choice(
                instructions="Which team should pick this up?",
                criteria={
                    "support": "How-to questions and account lookups.",
                    "infra": "Deploys, availability, and outages.",
                },
            ),
            "severity": Score(
                instructions="How severe is the impact?",
                criteria=["Cosmetic.", "Degraded for some users.", "Full outage."],
            ),
        },
    })
    print(
        label,
        "urgent", round(response.nouls["urgent"].noul, 2),
        "team", response.choices["team"].choice,
        "severity", response.scores["severity"].score,
    )


incident urgent 0.95 team infra severity 2.0
calm urgent 0.05 team support severity 0.0


## 3. Route the model for one run

`ModelRouterMiddleware` classifies the latest human message once, stores the `ChoiceAnswer` on `model_route`, and uses that model for every model call in the run. The two models here are scripted so you can see which one answered. Swap them for `init_chat_model("groq:...")` when you want real tokens. The middleware does not change.


In [3]:
def routed_agent(user_text: str, fast_reply: str, powerful_reply: str):
    fast = ScriptedChat(messages=iter([AIMessage(content=fast_reply)]))
    powerful = ScriptedChat(messages=iter([AIMessage(content=powerful_reply)]))
    router = ModelRouterMiddleware(
        classifier=classifier,
        choices={
            "fast": ModelChoice(model=fast, criteria="Short factual replies."),
            "powerful": ModelChoice(
                model=powerful,
                criteria="Architecture and multi-step reasoning.",
            ),
        },
        instructions="Pick powerful when the request is about architecture.",
    )
    agent = create_agent(fast, tools=[], middleware=[router])
    return agent.invoke({"messages": [HumanMessage(content=user_text)]})


simple = routed_agent("What is 2 + 2?", "fast-model", "powerful-model")
print(simple["model_route"].choice, simple["messages"][-1].content)

hard = routed_agent(
    "Design the architecture of a billing system",
    "fast-model",
    "powerful-model",
)
print(hard["model_route"].choice, hard["messages"][-1].content)


fast fast-model
powerful powerful-model


## 4. Block a risky tool call

`AutoModeMiddleware` asks a `Noul` question before the tools you name. A probability of 0.5 or more becomes an error `ToolMessage` and the tool function does not run. Other tools are left alone.

The local classifier treats the word `delete` as risky. A hosted classifier would read the tool arguments and the user's request instead of matching a keyword.


In [4]:
@tool
def delete_file(path: str) -> str:
    """Delete a file."""
    return f"deleted {path}"


@tool
def read_note(path: str) -> str:
    """Read a note."""
    return f"contents of {path}"


def run(scripted_messages, user_text: str):
    agent = create_agent(
        ScriptedChat(messages=iter(scripted_messages)),
        tools=[delete_file, read_note],
        middleware=[AutoModeMiddleware(tools=["delete_file"], classifier=classifier)],
    )
    result = agent.invoke({"messages": [HumanMessage(content=user_text)]})
    tool_messages = [message for message in result["messages"] if message.type == "tool"]
    return tool_messages[0]


blocked = run(
    [
        AIMessage(
            content="",
            tool_calls=[{
                "name": "delete_file",
                "args": {"path": "secret.txt"},
                "id": "c1",
                "type": "tool_call",
            }],
        ),
        AIMessage(content="stopped"),
    ],
    "please delete secret.txt",
)
print("delete status:", blocked.status)
print("delete ran:", blocked.content.startswith("deleted"))

allowed = run(
    [
        AIMessage(
            content="",
            tool_calls=[{
                "name": "read_note",
                "args": {"path": "a.txt"},
                "id": "c2",
                "type": "tool_call",
            }],
        ),
        AIMessage(content="read ok"),
    ],
    "read a.txt",
)
print("read result:", allowed.content)


delete status: error
delete ran: False
read result: contents of a.txt


## 5. Summary

- A decision model answers `Noul`, `Choice`, `Score`, and `Box` questions about one piece of state.
- Subclass `BaseDecisionModel` and implement `_classify` when you want a local or private classifier.
- `ModelRouterMiddleware` picks the model for a whole run and stores the answer on `model_route`.
- `AutoModeMiddleware` blocks named tool calls whose risk probability is at least 0.5.
- Both middleware classes are beta and take `classifier=`. `TypeSafeClassifier` is the hosted implementation.

`HumanInTheLoopMiddleware` (Tutorial 24) can also pause once per tool call with `interrupt_mode="per_call"`. Use that when a person must approve the call. Use `AutoModeMiddleware` when a classifier can refuse it.
